# GraphFusion — exploring an integration session

Drives the orchestrator directly (no UI, no LLM): load the sample datasets, inspect evidence, plan, merge and read provenance. Run from the repository root after `pip install -e .`.

In [ ]:
from pathlib import Path
from backend.session.state import Session

s = Session(workspace_root='workspace')
for f in ['sample_customers.csv', 'sample_customer_master.json', 'sample_sales.parquet']:
    s.add_file(Path('data/sample') / f)
summary = s.discover()
[(r['left'], r['right'], r['join_kind'], r['confidence']) for r in summary['relationships']]

In [ ]:
# evidence behind one correspondence
print(s.explain_match('city', 'location')['text'])

In [ ]:
# most reliable route between two datasets (Dijkstra on -log confidence)
print(s.route('sample_sales', 'sample_customer_master')['explanation'])

In [ ]:
plan = s.make_plan()
for step in plan.steps:
    print(step.step, step.operation, '-', step.description[:160])

In [ ]:
rec = s.execute(write_csv=False)
print(rec.result.quality_report['text'])

In [ ]:
import duckdb
duckdb.sql(f"SELECT customer_name, city, email, amount_spent_usd, _match_confidence FROM read_parquet('{Path(rec.result.output_path).as_posix()}') LIMIT 10").df()

In [ ]:
# conflicts are recorded with every candidate value
s.conflicts(limit=3)['conflicts']

In [ ]:
print(s.lineage_report()[:2000])